# ch09_dry_gas_production_systems — 9.3 Dewpoint thermodynamics

From chapter `ch09_dry_gas_production_systems`, section: **9.3 Dewpoint thermodynamics**.

This companion notebook reproduces or expands the textbook code block. Run all cells from top to bottom; cells that are intentionally illustrative are marked in the code comments.



In [1]:
from neqsim import jneqsim as ns
import numpy as np

gas = ns.thermo.system.SystemSrkEos(280.0, 70.0)
for c, x in [("methane", 0.92), ("ethane", 0.04),
             ("propane", 0.02), ("n-butane", 0.005),
             ("n-pentane", 0.001), ("nitrogen", 0.005),
             ("CO2", 0.009)]:
    gas.addComponent(c, x)
gas.setMixingRule("classic")

ops = ns.thermodynamicoperations.ThermodynamicOperations(gas)
ops.calcPTphaseEnvelope(True, 1.0)
env = ops.getOperation()

def finite_branch(temperatures, pressures):
   T = np.asarray(list(temperatures), dtype=float)
   P = np.asarray(list(pressures), dtype=float)
   keep = np.isfinite(T) & np.isfinite(P)
   return T[keep], P[keep]

branches = []
for label, get_T, get_P in [
   ("bubble-labelled", env.getBubblePointTemperatures, env.getBubblePointPressures),
   ("dew-labelled", env.getDewPointTemperatures, env.getDewPointPressures),
]:
   T, P = finite_branch(get_T(), get_P())
   if len(T) > 0:
      branches.append((label, T, P))

# Branch with higher max-T is the hydrocarbon dew curve (cricondentherm).
dew_label, dew_T, dew_P = max(branches, key=lambda item: item[1].max())
print(f"Hydrocarbon cricondentherm: {dew_T.max() - 273.15:.1f} °C ({dew_label})")
print(f"Cricondenbar: {max(P.max() for _, _, P in branches):.1f} bara")



Hydrocarbon cricondentherm: -31.8 °C (bubble-labelled)
Cricondenbar: 71.0 bara
